# Module 4 — Delivering Data Products with Snowflake

Everything you've built so far — the data model in Module 2, the transformations in Module 3 — needs to reach end users. Module 4 covers three progressively richer delivery mechanisms, all running on Snowflake compute with no data leaving the account:

1. **Inline Streamlit in a Notebook** — the fastest way to hand a stakeholder an interactive view of the data.
2. **Streamlit in Snowflake app** — the same visualization as a persistent, shareable Snowflake object.
3. **Snowflake Native App** — packaged and distributable via the Snowflake Marketplace, with consumer-driven references and versioning.

## Learning objectives
- Build an inline Streamlit visualization directly in a Snowflake Notebook cell.
- Deploy the same visualization as a first-class **Streamlit in Snowflake** app.
- Understand the components of a **Native App** — manifest, setup script, project definition, `snow app` CLI.

## Prerequisites
- Module 2 and Module 3 completed. The `tasty_bytes.harmonized.weather_hamburg` view (built in Module 3) must exist.
- For Section C only: Snowflake CLI installed locally, and the `modern-data-engineering-snowflake` repo cloned so you can run `snow app` from `module-4/snowflake-native-app/hamburg_weather_native_app/`.

## 1. Set your session context

In [ ]:
USE ROLE accountadmin;
USE WAREHOUSE compute_wh;
USE DATABASE tasty_bytes;
USE SCHEMA harmonized;

## Section A — Inline Streamlit visualization in the notebook

Snowflake Notebooks auto-import `streamlit as st`. Anything you can put in a Streamlit app, you can put in a Python cell. That means you can prototype an entire interactive data product without leaving the notebook.

The cell below builds a small multi-measure line chart of daily sales, temperature, precipitation, and wind speed for Hamburg — reading straight from `tasty_bytes.harmonized.weather_hamburg` (the view built in Module 3 with the F→C and in→mm UDFs applied).

In [ ]:
import streamlit as st
import altair as alt
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import col

session = get_active_session()

st.title("Weather and Sales Trends for Hamburg, Germany")

# Load the view built in Module 3
hamburg_weather = session.table("TASTY_BYTES.HARMONIZED.WEATHER_HAMBURG").select(
    col("DATE"),
    col("DAILY_SALES"),
    col("AVG_TEMPERATURE_FAHRENHEIT"),
    col("AVG_PRECIPITATION_INCHES"),
    col("MAX_WIND_SPEED_100M_MPH"),
).to_pandas()

# Melt to long form for a single multi-measure chart
long_df = hamburg_weather.melt("DATE", var_name="Measure", value_name="Value")
long_df["Measure"] = long_df["Measure"].replace({
    "DAILY_SALES": "Daily Sales ($)",
    "AVG_TEMPERATURE_FAHRENHEIT": "Avg Temperature (°F)",
    "AVG_PRECIPITATION_INCHES": "Avg Precipitation (in)",
    "MAX_WIND_SPEED_100M_MPH": "Max Wind Speed (mph)",
})

chart = (
    alt.Chart(long_df)
    .mark_line(point=True)
    .encode(
        x=alt.X("DATE:T", title="Date"),
        y=alt.Y("Value:Q", title="Values"),
        color=alt.Color(
            "Measure:N",
            title="Legend",
            scale=alt.Scale(range=["#29B5E8", "#FF6F61", "#0072CE", "#FFC300"]),
        ),
        tooltip=["DATE:T", "Measure:N", "Value:Q"],
    )
    .interactive()
    .properties(width=700, height=400, title="Daily Sales, Temperature, Precipitation, and Wind Speed in Hamburg")
)

st.altair_chart(chart, use_container_width=True)

Streamlit widgets work too. The next cell adds a date-range selector and a measure filter so a stakeholder can slice the same data interactively. This is the "prototype a data product without shipping anything" mode — a shareable notebook link is the whole distribution mechanism.

In [ ]:
import streamlit as st
import pandas as pd

min_date = pd.to_datetime(hamburg_weather["DATE"].min())
max_date = pd.to_datetime(hamburg_weather["DATE"].max())

date_range = st.date_input(
    "Date range",
    value=(min_date, max_date),
    min_value=min_date,
    max_value=max_date,
)

measures = st.multiselect(
    "Measures to show",
    options=["Daily Sales ($)", "Avg Temperature (°F)", "Avg Precipitation (in)", "Max Wind Speed (mph)"],
    default=["Daily Sales ($)", "Avg Temperature (°F)"],
)

if isinstance(date_range, tuple) and len(date_range) == 2 and measures:
    start, end = pd.to_datetime(date_range[0]), pd.to_datetime(date_range[1])
    filtered = long_df[
        (long_df["Measure"].isin(measures))
        & (long_df["DATE"] >= start)
        & (long_df["DATE"] <= end)
    ]
    st.line_chart(filtered.pivot(index="DATE", columns="Measure", values="Value"))
else:
    st.info("Pick a date range and at least one measure to see the chart.")

## Section B — Ship the same visualization as a Streamlit in Snowflake app

Section A is great for iteration, but stakeholders want a stable URL, a persistent app object, and access via `GRANT USAGE`. That's what **Streamlit in Snowflake** provides. The repo ships `module-4/streamlit-in-snowflake/streamlit_weather_hamburg.py` — the same visualization you just built inline, packaged as a standalone `.py`.

To deploy it, upload the file to a stage and create a `STREAMLIT` object that points at it. Below we create a scratch stage, then hand off to the terminal for the file upload (staging binary files from a notebook is awkward — the CLI is cleaner).

In [ ]:
CREATE OR REPLACE STAGE tasty_bytes.public.streamlit_apps
DIRECTORY = (ENABLE = TRUE);

### ⚠️ CLI handoff — upload the app file

From your terminal:

```bash
cd modern-data-engineering-snowflake/module-4/streamlit-in-snowflake

snow stage copy streamlit_weather_hamburg.py @tasty_bytes.public.streamlit_apps/hamburg/
```

Then come back to the notebook.

In [ ]:
CREATE OR REPLACE STREAMLIT tasty_bytes.public.streamlit_weather_hamburg
ROOT_LOCATION = '@tasty_bytes.public.streamlit_apps/hamburg'
MAIN_FILE = '/streamlit_weather_hamburg.py'
QUERY_WAREHOUSE = compute_wh
COMMENT = 'Hamburg weather + sales trends — Streamlit in Snowflake';

The `STREAMLIT` object is now a first-class Snowflake object. Open it in Snowsight (Projects → Streamlit → `STREAMLIT_WEATHER_HAMBURG`) and share it with `GRANT USAGE ON STREAMLIT ... TO ROLE <role>`.

Quick verification the object exists:

In [ ]:
SHOW STREAMLITS IN SCHEMA tasty_bytes.public;

## Section C — Package as a Snowflake Native App

Streamlit in Snowflake is great for internal delivery. To distribute a data product **outside** your account — to a partner, customer, or via the Marketplace — you need a **Snowflake Native App**.

The repo contains a complete Native App project at `module-4/snowflake-native-app/hamburg_weather_native_app/`:

| File | Purpose |
| --- | --- |
| `snowflake.yml` | Project definition used by the `snow app` CLI |
| `app/manifest.yml` | Manifest describing the app version, artifacts, and references |
| `app/setup_script.sql` | Setup script run when a consumer installs the app |
| `app/ui/streamlit.py` | The Streamlit UI, using `permission.request_reference()` to ask consumers for a table reference |
| `scripts/setup-package-script.sql` | Package-side setup (runs on the provider side) |

Below we read the manifest and setup script inline so you can see the shape — but the actual deployment happens in the terminal via `snow app`.

In [ ]:
import os

repo_root = "/opt/notebook-repo"  # placeholder — replace with your local path if reading files
manifest_path = "module-4/snowflake-native-app/hamburg_weather_native_app/app/manifest.yml"

# Display the manifest inline so students see the shape without switching to a file browser.
# (This cell reads from the notebook's working directory — if you're running in Snowsight,
#  paste the manifest contents below or use a Git-backed workspace so the file is available.)
manifest = """
manifest_version: 1

version:
  name: hamburg_app_v1
  label: hamburg_app_v1_label
  comment: The first version of the application

artifacts:
  setup_script: setup_script.sql
  default_streamlit: app_instance_schema.streamlit
  extension_code: true

configuration:
  log_level: debug
  trace_level: off

references:
  - weather_hamburg_app:
      label: "WEATHER_HAMBURG_T"
      description: "SELECT TABLE"
      privileges:
        - SELECT
      object_type: Table
      multi_valued: false
      register_callback: app_instance_schema.update_reference
"""

print(manifest)

### Manifest walkthrough

- **`version`** — declares an application version. Consumers install/upgrade by version.
- **`artifacts.setup_script`** — the SQL script run in the consumer account when the app is installed.
- **`artifacts.default_streamlit`** — the Streamlit object to launch when a consumer opens the app.
- **`references`** — **the interesting part.** The app doesn't ship any data. Instead, it declares a `weather_hamburg_app` reference that the consumer *binds* to a table in their own account (via a callback proc `update_reference`). This is how a Native App consumes data without the provider ever seeing it.

### Setup script walkthrough

The setup script (`app/setup_script.sql`) does four things at install time:

1. Creates an **application role** (`app_instance_role`) and a **versioned schema** (`app_instance_schema`).
2. Creates a view over `shared_content_schema.weather_hamburg_app` — the schema the provider seeds on the package side.
3. Creates the Streamlit app from `/ui/streamlit.py`.
4. Creates the `update_reference` procedure that Snowflake calls when the consumer binds a table to the `weather_hamburg_app` reference.

Together these give the consumer a working app the moment they install.

### ⚠️ CLI handoff — deploy the Native App

Native App projects are a filesystem-first concept — everything drives off `snowflake.yml` in the project root. Deploy with the `snow app` CLI:

```bash
cd modern-data-engineering-snowflake/module-4/snowflake-native-app/hamburg_weather_native_app

# Build the application package + install the app in your account
snow app run

# Open the app in Snowsight
snow app open
```

When the app opens for the first time in Snowsight, you'll be prompted to bind the `weather_hamburg_app` reference. Point it at `tasty_bytes.harmonized.weather_hamburg` (or the `weather_hamburg_t` table if you've materialized one). The Streamlit UI will render the same Hamburg chart, this time inside an installed application.

For details on every `snow app` sub-command, see the [Snowflake CLI docs](https://docs.snowflake.com/en/developer-guide/snowflake-cli-v2/plugins/app/index).

Optional teardown from the terminal:

```bash
snow app teardown
```

## Recap

You practiced the three delivery patterns Snowflake supports:

| Pattern | Where it lives | Best for |
|---|---|---|
| **Inline Streamlit in a Notebook** | Inside a notebook cell | Prototyping, quick data products, one-off shareable links |
| **Streamlit in Snowflake** | Standalone `STREAMLIT` object in your account | Internal, persistent, GRANT-able apps |
| **Snowflake Native App** | An installable Application Package | External distribution, versioned releases, Marketplace publishing |

All three run entirely on Snowflake compute. None of them move your data out. Module 5 automates the pipeline that keeps these data products fresh.